# 00_Data cleaning

Cleans the raw MyAnimeList-based Anime Dataset 2026 (30,763 titles, 24 columns) into an analysis-ready table. Placeholder values are converted to proper missing values, text fields are parsed into usable features, and a recommendation catalog is defined for the later notebooks.

Output: `data/processed/anime_clean.parquet`

## 1. Load data

In [1]:
import re

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

raw_path = '../data/raw/anime-dataset-2026.csv'
processed_path = '../data/processed/anime_clean.parquet'

df = pd.read_csv(raw_path)
print(df.shape)
df.head(3)

(30763, 24)


,anime_id,Name,English name,Other name,Score,Genres,Synopsis,Type,Episodes,Aired,Premiered,Status,Producers,Licensors,Studios,Source,Duration,Rating,Rank,Popularity,Favorites,Scored By,Members,Image URL
0,1,Cowboy Bebop,Cowboy Bebop,カウボーイビバップ,8.75,"Action, Award Winning, Sci-Fi, Adult Cast, Space","Crime is timeless. By the year 2071, humanity ...",TV,26,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,Finished Airing,"Bandai Visual, Victor Entertainment, Audio Pla...",Funimation,Sunrise,Original,24 min. per ep.,R - 17+ (violence & profanity),50,41,90976,1077271,2089081,https://cdn.myanimelist.net/images/anime/4/196...
1,5,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop: The Movie,カウボーイビバップ 天国の扉,8.38,"Action, Sci-Fi, Adult Cast, Space","Another day, another bounty—such is the life o...",Movie,1,"Sep 1, 2001",UNKNOWN,Finished Airing,"Sunrise, Bandai Visual","Sony Pictures Entertainment, Funimation",Bones,Original,1 hr. 55 min.,R - 17+ (violence & profanity),252,663,1827,234731,417292,https://cdn.myanimelist.net/images/anime/1439/...
2,6,Trigun,Trigun,トライガン,8.22,"Action, Adventure, Sci-Fi, Adult Cast, Shounen","Vash the Stampede is the man with a $$60,000,0...",TV,26,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,Finished Airing,"Victor Entertainment, Arts Pro",Funimation,Madhouse,Manga,24 min. per ep.,PG-13 - Teens 13 or older,433,268,17956,405835,846067,https://cdn.myanimelist.net/images/anime/1130/...


## 2. Convert placeholders to missing values

The raw file stores missing values as the string `UNKNOWN`, and missing synopses as a fixed placeholder sentence. Both are replaced with `NaN` so pandas treats them as missing.

In [2]:
df = df.replace('UNKNOWN', np.nan)

placeholder = df['Synopsis'].str.contains('No synopsis', case=False, na=False)
df.loc[placeholder, 'Synopsis'] = np.nan
print(f'Synopsis placeholders removed: {placeholder.sum():,}')

missing = (df.isna().mean() * 100).sort_values(ascending=False).round(1)
missing.to_frame('missing_%')

Synopsis placeholders removed: 5,040


,missing_%
Licensors,82.9
Premiered,78.5
English name,55.3
Producers,52.5
Studios,39.5
Score,33.8
Scored By,33.8
Rank,25.0
Synopsis,16.4
Episodes,2.5


## 3. Numeric columns

Score, episode count and the engagement counts are stored as text because of the placeholders. They are converted to numbers here.

In [3]:
num_cols = ['Score', 'Episodes', 'Rank', 'Popularity', 'Favorites', 'Scored By', 'Members']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df[num_cols].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Score,20358.0,6.41,0.89,1.89,5.79,6.39,7.04,9.25
Episodes,30007.0,13.78,46.90,1.00,1.00,2.00,13.00,3000.00
Rank,23067.0,11378.60,6658.59,0.00,5612.50,11375.00,17146.00,22912.00
Popularity,30763.0,15227.87,8880.87,0.00,7535.50,15227.00,22918.00,30608.00
Favorites,30763.0,429.17,4582.16,0.00,0.00,1.00,17.00,256516.00
Scored By,20358.0,29742.40,123518.04,102.00,311.00,1360.00,9575.75,3115782.00
Members,30763.0,39192.86,170966.33,0.00,241.00,1051.00,9162.00,4434071.00


## 4. Air date

`Premiered` is missing for most titles, so the start year and season are parsed from `Aired` instead (e.g. `Apr 3, 1998 to Apr 24, 1999`). Only the start date is kept.

In [4]:
start = df['Aired'].str.split(' to ').str[0].str.strip()

df['year'] = pd.to_numeric(start.str.extract(r'(\d{4})')[0], errors='coerce')

month_map = {m: i for i, m in enumerate(
    ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], start=1)}
month = start.str.extract(r'^([A-Z][a-z]{2})')[0].map(month_map)

season_map = {12: 'Winter', 1: 'Winter', 2: 'Winter',
              3: 'Spring', 4: 'Spring', 5: 'Spring',
              6: 'Summer', 7: 'Summer', 8: 'Summer',
              9: 'Fall', 10: 'Fall', 11: 'Fall'}
df['season'] = month.map(season_map)

print(f"Year parsed: {df['year'].notna().mean():.1%}")
print(f"Season parsed: {df['season'].notna().mean():.1%}")
df[['Aired', 'year', 'season']].sample(5, random_state=42)

Year parsed: 97.1%
Season parsed: 85.3%


,Aired,year,season
12594,"Apr 14, 2013",2013.0,Spring
5803,"Aug 21, 1996",1996.0,Summer
22884,"Aug 14, 2022",2022.0,Summer
7555,"Oct 7, 1987",1987.0,Fall
23254,"Jun 1, 2022 to Jul 20, 2022",2022.0,Summer


## 5. Episode duration

`Duration` is free text such as `24 min. per ep.` or `1 hr. 55 min.`. It is converted to minutes per episode.

In [5]:
def parse_duration(s):
    if pd.isna(s):
        return np.nan
    s = str(s).lower()
    hr = re.search(r'(\d+)\s*hr', s)
    mn = re.search(r'(\d+)\s*min', s)
    sec = re.search(r'(\d+)\s*sec', s)
    if not (hr or mn or sec):
        return np.nan
    return ((int(hr.group(1)) * 60 if hr else 0)
            + (int(mn.group(1)) if mn else 0)
            + (int(sec.group(1)) / 60 if sec else 0))

df['duration_min'] = df['Duration'].apply(parse_duration)
df['duration_min'].describe().round(1)

count    29872.0
mean        20.6
std         24.2
min          0.0
25%          4.0
50%         14.0
75%         24.0
max        168.0
Name: duration_min, dtype: float64

## 6. Multi-value columns

Genres, studios, producers and licensors are comma-separated strings. They are split into lists so each value can be counted or encoded separately.

In [6]:
def split_multi(s):
    if pd.isna(s):
        return []
    return [x.strip() for x in str(s).split(',') if x.strip()]

multi_cols = {'Genres': 'genres', 'Studios': 'studios',
              'Producers': 'producers', 'Licensors': 'licensors'}
for raw_col, new_col in multi_cols.items():
    df[new_col] = df[raw_col].apply(split_multi)

df['n_genres'] = df['genres'].str.len()
df[['Name', 'genres', 'studios', 'licensors']].head()

,Name,genres,studios,licensors
0,Cowboy Bebop,"[Action, Award Winning, Sci-Fi, Adult Cast, Sp...",[Sunrise],[Funimation]
1,Cowboy Bebop: Tengoku no Tobira,"[Action, Sci-Fi, Adult Cast, Space]",[Bones],"[Sony Pictures Entertainment, Funimation]"
2,Trigun,"[Action, Adventure, Sci-Fi, Adult Cast, Shounen]",[Madhouse],[Funimation]
3,Witch Hunter Robin,"[Action, Drama, Mystery, Supernatural, Detective]",[Sunrise],"[Funimation, Bandai Entertainment]"
4,Bouken Ou Beet,"[Action, Adventure, Fantasy, Shounen]",[Toei Animation],[Illumitoon Entertainment]


## 7. Licensing flag

On MyAnimeList, titles without an English-market licensor show "None found", which this dataset appears to record as `UNKNOWN`. If so, an empty licensor list means *not licensed* rather than *unknown*. The check below lists the most-watched titles without a licensor; if most of them are known to be licensed, the flag is unreliable.

In [7]:
df['is_licensed'] = (df['licensors'].str.len() > 0).astype(int)
print(f"Share with a licensor: {df['is_licensed'].mean():.1%}")

(df.loc[df['is_licensed'] == 0, ['Name', 'Type', 'year', 'Members']]
   .sort_values('Members', ascending=False)
   .head(15))

Share with a licensor: 17.1%


,Name,Type,year,Members
20942,Sono Bisque Doll wa Koi wo Suru,TV,2022.0,1375058
6596,Kuroko no Basket,TV,2012.0,1331278
15549,Re:Zero kara Hajimeru Isekai Seikatsu 2nd Season,TV,2020.0,1323379
12414,Nanatsu no Taizai: Imashime no Fukkatsu,TV,2018.0,1223249
22637,Ore dake Level Up na Ken,TV,2024.0,1194558
21800,Spy x Family Part 2,TV,2022.0,1163830
13238,Wotaku ni Koi wa Muzukashii,TV,2018.0,1120824
16824,Re:Zero kara Hajimeru Isekai Seikatsu 2nd Seas...,TV,2021.0,1112506
16240,"Kanojo, Okarishimasu",TV,2020.0,1105443
21028,"Komi-san wa, Comyushou desu.",TV,2021.0,1033884


## 8. Recommendation catalog

The recommender should only suggest titles a viewer could reasonably pick up. A title is kept in the catalog if it:

- is a TV series, movie or ONA (original net animation, the format used by most streaming originals)
- has a score from at least 1,000 users, so the score is meaningful
- has a synopsis, since the story text is a core content feature

In [8]:
MIN_SCORED_BY = 1_000
KEEP_TYPES = ['TV', 'Movie', 'ONA']

steps = [
    ('All titles', pd.Series(True, index=df.index)),
    ('TV / Movie / ONA', df['Type'].isin(KEEP_TYPES)),
    ('Has a score', df['Score'].notna()),
    (f'Scored by >= {MIN_SCORED_BY:,} users', df['Scored By'] >= MIN_SCORED_BY),
    ('Has a synopsis', df['Synopsis'].notna()),
]

mask = pd.Series(True, index=df.index)
funnel = []
for label, cond in steps:
    mask &= cond
    funnel.append((label, mask.sum()))

df['in_catalog'] = mask.astype(int)
pd.DataFrame(funnel, columns=['step', 'titles'])

,step,titles
0,All titles,30763
1,TV / Movie / ONA,18307
2,Has a score,11223
3,"Scored by >= 1,000 users",6829
4,Has a synopsis,6801


## 9. Save

In [9]:
keep = ['anime_id', 'Name', 'English name', 'Score', 'Synopsis', 'Type', 'Episodes',
        'Source', 'Rating', 'Rank', 'Popularity', 'Favorites', 'Scored By', 'Members',
        'Image URL', 'year', 'season', 'duration_min',
        'genres', 'studios', 'producers', 'licensors', 'n_genres',
        'is_licensed', 'in_catalog']

clean = df[keep].rename(columns={'English name': 'english_name',
                                 'Scored By': 'scored_by',
                                 'Image URL': 'image_url'})
clean.columns = [c.lower() for c in clean.columns]

clean.to_parquet(processed_path, index=False)
print(clean.shape, '->', processed_path)

(30763, 25) -> ../data/processed/anime_clean.parquet


## Key findings

- **Missing data is heavy outside the core fields.** `Licensors` (82.9%), `Premiered` (78.5%), `Producers` (52.5%) and `Studios` (39.5%) are mostly missing. A third of titles (33.8%) have no score; these are almost all obscure or unreleased entries. 5,040 synopses (16.4%) were placeholder text and are now treated as missing.
- **`Aired` is a usable replacement for `Premiered`.** The start year parses for 97.1% of titles and the season for 85.3%, versus 21.5% coverage in `Premiered`.
- **The licensing field cannot be used.** Only 17.1% of titles have a licensor, but the most-watched titles without one include clearly licensed series such as *Spy x Family Part 2*, *Blue Lock*, *Ore dake Level Up na Ken* (Solo Leveling) and *Kuroko no Basket*. A missing licensor therefore means *unknown*, not *unlicensed*. `is_licensed` is kept in the file for transparency but is not used in later notebooks.
- **Engagement is extremely concentrated.** Median members is 1,051 while the maximum is 4.4M, and the median `Scored By` is 1,360 against a mean of 29,742. Counts need a log scale in EDA and must not dominate similarity in the recommender.
- **The recommendation catalog has 6,801 titles.** Of 30,763 entries, 18,307 are TV, movies or ONAs; 11,223 of those have a score; 6,829 are scored by at least 1,000 users; and 6,801 also have a synopsis.